In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import models, layers

In [2]:
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)

In [3]:
train_ds, val_ds = tf.keras.utils.audio_dataset_from_directory(
    directory='dataset',
    seed=seed,
    batch_size=64, 
    output_sequence_length=16000,
    validation_split=0.3,
    subset='both'
)

Found 6505 files belonging to 4 classes.
Using 4554 files for training.
Using 1951 files for validation.


In [4]:
label = np.array(train_ds.class_names)
# print(label)

In [5]:
def squeeze(audio, label):
    audio = tf.squeeze(audio, axis=-1)
    return audio, label

def get_st(wf):
    st = tf.signal.stft(wf, 255, 128)
    st = tf.abs(st)
    st = st[..., tf.newaxis]
    return st

def make_st_ds(ds):
    return ds.map(
        lambda i, j: (get_st(i), j),
        tf.data.AUTOTUNE
    )

In [6]:
train_ds = train_ds.map(squeeze, tf.data.AUTOTUNE)
val_ds = val_ds.map(squeeze, tf.data.AUTOTUNE)

test_ds = val_ds.shard(num_shards=2, index=0)
val_ds = val_ds.shard(num_shards=2, index=1)

In [7]:
train_st_ds = make_st_ds(train_ds)
val_st_ds = make_st_ds(val_ds)
test_st_ds = make_st_ds(test_ds)

In [11]:
for i, j in train_st_ds.take(1):
    break

input_shape = i.shape[1:]
label_len = len(label)

# print(input_shape)
# print(label_len)

In [13]:
norm_layer = layers.Normalization()
norm_layer.adapt(
    data=train_st_ds.map(
        lambda i, j: i
    )
)

In [15]:
model = models.Sequential([
    layers.Input(input_shape),
    layers.Resizing(32, 32),
    norm_layer,
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.5),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(label_len)
])

In [17]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resizing_1 (Resizing)       (None, 32, 32, 1)         0         
                                                                 
 normalization_2 (Normalizat  (None, 32, 32, 1)        3         
 ion)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 30, 30, 64)        640       
                                                                 
 max_pooling2d_2 (MaxPooling  (None, 15, 15, 64)       0         
 2D)                                                             
                                                                 
 conv2d_3 (Conv2D)           (None, 13, 13, 128)       73856     
                                                                 
 max_pooling2d_3 (MaxPooling  (None, 6, 6, 128)        0

In [20]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [21]:
history = model.fit(
    train_st_ds,
    validation_data=val_st_ds,
    epochs=10,
    callbacks=tf.keras.callbacks.EarlyStopping(verbose=1, patience=2)
)

Epoch 1/10
72/72 [==============================] - 6s 72ms/step - loss: 0.9103 - accuracy: 0.6436 - val_loss: 0.5368 - val_accuracy: 0.7833
Epoch 2/10
72/72 [==============================] - 5s 72ms/step - loss: 0.5205 - accuracy: 0.8195 - val_loss: 0.3721 - val_accuracy: 0.8719
Epoch 3/10
72/72 [==============================] - 6s 73ms/step - loss: 0.3956 - accuracy: 0.8652 - val_loss: 0.2740 - val_accuracy: 0.9083
Epoch 4/10
72/72 [==============================] - 5s 72ms/step - loss: 0.3290 - accuracy: 0.8911 - val_loss: 0.2318 - val_accuracy: 0.9229
Epoch 5/10
72/72 [==============================] - 6s 73ms/step - loss: 0.2748 - accuracy: 0.9040 - val_loss: 0.2191 - val_accuracy: 0.9271
Epoch 6/10
72/72 [==============================] - 6s 73ms/step - loss: 0.2438 - accuracy: 0.9141 - val_loss: 0.1936 - val_accuracy: 0.9292
Epoch 7/10
72/72 [==============================] - 6s 73ms/step - loss: 0.2148 - accuracy: 0.9234 - val_loss: 0.1891 - val_accuracy: 0.9365
Epoch 8/10
72

In [23]:
for i, (spec_batch, lab_batch) in enumerate(test_st_ds):
    pred = model.predict(spec_batch)

    for j in range(len(lab_batch)):
        t = lab_batch[j].numpy()
        p = np.argmax(pred[j])

        print(f'{t}, {p}')

2/2 [==============================] - 0s 82ms/step
2, 2
1, 1
2, 2
0, 0
2, 2
0, 0
0, 0
0, 0
0, 0
1, 1
3, 3
1, 1
2, 2
0, 0
0, 0
2, 2
1, 2
0, 0
3, 3
1, 3
1, 1
2, 2
1, 1
1, 1
3, 3
2, 2
2, 2
0, 0
0, 0
1, 1
0, 0
3, 3
3, 3
1, 1
1, 1
0, 0
0, 0
3, 3
1, 1
2, 2
3, 3
2, 2
2, 2
3, 3
1, 1
2, 2
3, 3
1, 1
0, 0
1, 1
2, 2
3, 3
2, 2
2, 2
0, 0
3, 3
3, 3
2, 2
0, 0
3, 3
2, 2
0, 0
1, 1
1, 1
2/2 [==============================] - 0s 39ms/step
3, 3
2, 2
1, 1
2, 2
0, 0
3, 3
0, 0
2, 2
3, 3
3, 3
3, 3
0, 0
0, 0
1, 1
1, 1
0, 0
3, 3
0, 0
3, 3
1, 1
2, 2
0, 0
0, 0
0, 0
1, 1
1, 3
3, 3
1, 1
0, 0
0, 0
3, 3
0, 0
2, 2
0, 0
1, 1
2, 2
1, 1
1, 1
0, 2
1, 1
2, 2
2, 2
1, 1
2, 2
1, 1
0, 0
2, 2
0, 0
3, 3
1, 1
0, 0
3, 3
1, 1
1, 1
0, 0
3, 3
0, 0
0, 0
2, 2
3, 3
1, 1
3, 3
0, 0
0, 0
2/2 [==============================] - 0s 39ms/step
0, 0
3, 3
1, 1
0, 0
3, 3
3, 3
0, 0
3, 3
0, 0
2, 2
2, 2
2, 2
1, 1
1, 1
2, 2
3, 3
0, 2
0, 0
1, 1
1, 1
0, 0
2, 2
2, 2
0, 0
0, 0
3, 3
1, 1
3, 3
3, 3
0, 0
1, 1
0, 0
0, 0
0, 0
3, 3
0, 0
2, 2
3, 3
0, 0
1, 1
0, 0